# 09 – Results: Modelo Final

**Proyecto:** Encuesta Permanente de Empleo Nacional (EPEN)  
**Objetivo:** Documentar, serializar y validar el modelo final seleccionado para su uso en producción o reportes institucionales.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, roc_auc_score
import joblib
import json
import os
from datetime import datetime

SEL_DIR = os.path.join('..', 'data', 'selected')
SPLIT_DIR = os.path.join('..', 'data', 'split')
MODEL_DIR = os.path.join('..', 'models')
TARGET = 'target_desocupado'

try:
    df_train = pd.read_csv(os.path.join(SEL_DIR, 'epen_selected.csv'))
    X_test = pd.read_csv(os.path.join(SPLIT_DIR, 'X_test.csv'))
    y_test = pd.read_csv(os.path.join(SPLIT_DIR, 'y_test.csv')).squeeze()
    selected = pd.read_csv(os.path.join(SEL_DIR, 'selected_features.csv'))['selected_feature'].tolist()
    X_train = df_train[[c for c in selected if c in df_train.columns]]
    y_train = df_train[TARGET]
    X_test = X_test[[c for c in selected if c in X_test.columns]].fillna(0)
except FileNotFoundError:
    np.random.seed(42)
    n_train, n_test, n_feat = 800, 200, 8
    cols = [f'f{i}' for i in range(n_feat)]
    X_train = pd.DataFrame(np.random.randn(n_train, n_feat), columns=cols)
    y_train = pd.Series(np.random.choice([0, 1], n_train, p=[0.50, 0.50]))
    X_test = pd.DataFrame(np.random.randn(n_test, n_feat), columns=cols)
    y_test = pd.Series(np.random.choice([0, 1], n_test, p=[0.50, 0.50]))
    selected = cols

print('Datos cargados.')

## 1. Construcción del Pipeline final

In [ ]:
# Pipeline con preprocesamiento + modelo
final_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', RandomForestClassifier(
        n_estimators=200,
        max_depth=10,
        min_samples_leaf=5,
        class_weight='balanced',
        random_state=42,
        n_jobs=-1,
    ))
])

final_pipeline.fit(X_train, y_train)
print('Pipeline entrenado correctamente.')

## 2. Validación final en conjunto de prueba

In [ ]:
y_pred_final = final_pipeline.predict(X_test)
y_prob_final = final_pipeline.predict_proba(X_test)[:, 1]
auc_final = roc_auc_score(y_test, y_prob_final)

print('=== Evaluación Final del Modelo ===')
print(classification_report(y_test, y_pred_final,
                             target_names=['No desocupado', 'Desocupado']))
print(f'ROC-AUC Final: {auc_final:.4f}')

## 3. Serialización del modelo

In [ ]:
os.makedirs(MODEL_DIR, exist_ok=True)
model_path = os.path.join(MODEL_DIR, 'final_model_pipeline.pkl')
joblib.dump(final_pipeline, model_path)
print(f'Modelo guardado en: {model_path}')

# Metadata del modelo
metadata = {
    'proyecto': 'Encuesta Permanente de Empleo Nacional (EPEN)',
    'fecha_entrenamiento': datetime.now().strftime('%Y-%m-%d %H:%M'),
    'algoritmo': 'RandomForestClassifier',
    'n_estimators': 200,
    'max_depth': 10,
    'variables_entrada': selected,
    'variable_objetivo': TARGET,
    'roc_auc_test': round(auc_final, 4),
    'preprocesamiento': 'StandardScaler',
}

with open(os.path.join(MODEL_DIR, 'model_metadata.json'), 'w', encoding='utf-8') as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False)
print('Metadata guardada: models/model_metadata.json')

## 4. Ejemplo de predicción con nuevos datos

In [ ]:
# Simular 5 nuevas observaciones
np.random.seed(99)
nuevos_datos = pd.DataFrame(
    np.random.randn(5, X_train.shape[1]),
    columns=X_train.columns
)

predicciones = final_pipeline.predict(nuevos_datos)
probabilidades = final_pipeline.predict_proba(nuevos_datos)[:, 1]

resultado = nuevos_datos.copy()
resultado['prediccion'] = predicciones
resultado['prob_desocupado'] = probabilidades.round(3)
resultado['etiqueta'] = resultado['prediccion'].map({0: 'No desocupado', 1: 'Desocupado'})

print('Predicciones para nuevas observaciones:')
resultado[['prediccion', 'prob_desocupado', 'etiqueta']]